# Panama National Electricity Load — processing

Source: [kausnz/panama-electricity-load-forecast](https://github.com/kausnz/panama-electricity-load-forecast) (`continuous_dataset.csv`), originally from Panama's grid operator (CND) plus weather for 3 cities, published on [Mendeley Data](https://data.mendeley.com/datasets/byx7sztj59/1).

The cleanest of the 8 candidates on first inspection: zero missing values in any column, perfectly regular hourly index, 5.5 years of data (truncated to ~5.16 years below to remove the COVID-19 period -- see that section).

`Holiday_ID` (23 distinct integer codes identifying which specific holiday) was dropped in favor of the binary `holiday` flag: feeding a nominal ID as a raw integer would imply a false ordinal relationship, and one-hot encoding 23 mostly-empty categories was judged not worth the sparsity for this study. `school` (in-session flag) is kept as-is.

In [ ]:
import sys
sys.path.append(".")
import pandas as pd
from common import report_candidate

RAW_PATH = "../data/raw/panama_load.csv"
PROCESSED_PATH = "../data/processed/panama_load.csv"

df = pd.read_csv(RAW_PATH, parse_dates=["datetime"]).set_index("datetime")
df.index.name = "timestamp"
df.isna().sum()

In [ ]:
df = df.drop(columns=["Holiday_ID"])
df = df.rename(columns={"nat_demand": "national_demand_mw"})

TARGET = "national_demand_mw"
feature_cols = [c for c in df.columns if c != TARGET]
print(len(feature_cols), feature_cols)

## COVID-19 truncation

The chronological 80/20 train/test split lands the test set at 2019-05-23 -> 2020-06-27, which includes Panama's COVID-19 lockdown (a documented, severe demand shock -- April 2020 averaged 1062 MW vs 1214-1278 MW in April of every other year 2016-2019, with peak demand roughly halved). Training data (2015-2019) contains no precedent for this regime, so a model cannot legitimately learn to predict it, and evaluating on it doesn't measure forecasting skill -- it measures how a normal-times model degrades under an unmodeled exogenous shock, which is a different question.

Rather than carry that confound through every phase of the study, the dataset is truncated at 2020-03-01 (a round boundary, not the exact lockdown declaration date -- revisit if this matters for the final report). This drops the last ~4 months (2,833 of 48,048 rows, ~5.9%), leaving 45,215 rows / ~5.16 years, still ample for the 168h-window experiments.

In [ ]:
COVID_START = "2020-03-01"

before = len(df)
df = df[df.index < COVID_START]
print(f"dropped {before - len(df)} rows at/after {COVID_START} -> {len(df)} rows remain")
print(f"range now: {df.index.min()} -> {df.index.max()}")

In [ ]:
report_candidate(df, TARGET, feature_cols, freq="1h", name="Panama National Electricity Load (processed)")

In [ ]:
df.to_csv(PROCESSED_PATH)
print(f"saved: {PROCESSED_PATH}  shape={df.shape}")